In [ ]:
#| default_exp apple

# apple

> `.mlpackage` and `.mlmodel` over Core ML, on the Neural Engine where there is one.

`pip install 'anya[coreml]'`, macOS only. Core ML differs from the other two runtimes in one way
that matters: an image input is a `PIL.Image` at a size the model fixes, not a normalised tensor, so
most of `Prep` is the model's job rather than anya's. A Core ML classifier also returns its class
names with its scores, so there is no labels file to find.

Apple Foundation Models are language models and belong to `rishi`, not here. This module is Core ML:
the classifiers, detectors and segmenters Apple and the community ship as `.mlpackage`.

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path

import numpy as np
from fastcore.all import AttrDict, L

from anya.core import Model, Pred, infer_task, model_file, prep_from_spec, with_hub_defaults
from anya.vision import Prep, decode_classify, read_labels

In [ ]:
#| hide
from fastcore.test import *

## Reading a Core ML signature

`ct_spec` turns the protobuf description into the same `(name, shape, dtype)` rows the other
runtimes produce, so `infer_task` and `prep_from_spec` work unchanged.

In [ ]:
#| export
def _feat(f) -> AttrDict:
    'One input or output feature of a Core ML model as a plain row.'
    t = f.type.WhichOneof('Type')
    if t == 'imageType':
        return AttrDict(name=f.name, kind='image', shape=[1, int(f.type.imageType.height),
                                                          int(f.type.imageType.width), 3], dtype='uint8',
                        color=f.type.imageType.colorSpace)
    if t == 'multiArrayType':
        return AttrDict(name=f.name, kind='array', shape=[1] + [int(d) for d in f.type.multiArrayType.shape],
                        dtype='float32')
    return AttrDict(name=f.name, kind=t or 'unknown', shape=[], dtype='float32')

def ct_spec(mlmodel) -> AttrDict:
    'Input and output rows for a loaded `coremltools` model.'
    d = mlmodel.get_spec().description
    return AttrDict(inputs=L(d.input).map(_feat), outputs=L(d.output).map(_feat),
                    kind=mlmodel.get_spec().WhichOneof('Type'))

In [ ]:
#| export
def decode_class_dict(d:dict,           # a Core ML classifier's {label: probability}
                      topk:int=5
                     ) -> list:
    'Rank a Core ML classifier probability dictionary the way `decode_classify` ranks a tensor.'
    xs = sorted(d.items(), key=lambda t: -float(t[1]))[:max(1, topk)]
    return [dict(label=str(k), score=round(float(v), 6), index=i) for i, (k, v) in enumerate(xs)]

In [ ]:
#| hide
test_eq(decode_class_dict({'wren': 0.2, 'galah': 0.7, 'emu': 0.1}, topk=2),
        [dict(label='galah', score=0.7, index=0), dict(label='wren', score=0.2, index=1)])

## CoreMLModel

`compute_units` is the one knob worth knowing: `'ALL'` uses the Neural Engine, `'CPU_ONLY'` is what
you want when comparing numbers against another runtime.

In [ ]:
#| export
class CoreMLModel(Model):
    'A Core ML model as a `Model`. Image inputs go in as PIL images at the size the model fixes.'
    _runtime = 'coreml'

    def __init__(self,
                 model=None,               # a path to .mlpackage/.mlmodel, or a hub repo id
                 *,
                 runtime:str=None,
                 model_path=None,
                 file:str=None,
                 revision:str=None,
                 task:str=None,
                 labels=None,
                 norm=None,
                 size:tuple=None,
                 resize:str=None,
                 prep=None,
                 topk:int=5, conf:float=0.25, iou:float=0.45,
                 compute_units:str='ALL',  # 'ALL', 'CPU_ONLY', 'CPU_AND_GPU', 'CPU_AND_NE'
                 mlmodel=None,             # an already-loaded coremltools model
                 **kw):
        model = self._setup(model, task=task, labels=labels, topk=topk, conf=conf, iou=iou)
        self.model_path = str(model_file(model, model_path, file=file, revision=revision))
        self._sess = mlmodel or self._mk_model(compute_units, **kw)
        s = ct_spec(self._sess)
        self.inputs, self.outputs, self.kind = s.inputs, s.outputs, s.kind
        if len(self.inputs) > 1: raise ValueError(
            f'{Path(self.model_path).name} takes {len(self.inputs)} inputs; anya drives single-input models.')
        self.inp = self.inputs[0]
        labels, pk = with_hub_defaults(self.model_path, self.labels, norm=norm, size=size, resize=resize)
        self.labels = read_labels(labels)
        self._task = task or ('classify' if self.kind == 'neuralNetworkClassifier'
                              else infer_task([o.shape for o in self.outputs], self.labels,
                                              [o.name for o in self.outputs]))
        # an image input is resized and handed over as pixels: Core ML owns the normalisation
        self._prep = prep or (Prep(size=tuple(self.inp.shape[1:3]), layout='nhwc', dtype='uint8',
                                   resize=pk.get('resize') or ('letterbox' if self._task == 'detect' else 'stretch'),
                                   crop_pct=pk.get('crop_pct'))
                              if self.inp.kind == 'image' else
                              prep_from_spec(self.inp.shape, self.inp.dtype, task=self._task, **pk))
        self._max_bs = 1

    def _mk_model(self, compute_units='ALL', **kw):
        try: import coremltools as ct
        except ImportError as e:
            raise ImportError("anya needs coremltools for Core ML models: pip install 'anya[coreml]'") from e
        return ct.models.MLModel(self.model_path, compute_units=ct.ComputeUnit[compute_units], **kw)

    @property
    def spec(self) -> AttrDict:
        'What the model declares, as plain dicts.'
        return AttrDict(inputs=list(self.inputs), outputs=list(self.outputs), kind=self.kind)

    def _infer(self, x) -> list:
        if self._sess is None: raise RuntimeError('this model is closed')
        out = self._sess.predict({self.inp.name: self._as_input(x[0])})
        self._last = out
        return [np.asarray(out[o.name]) for o in self.outputs if o.name in out and not isinstance(out[o.name], dict)]

    def _as_input(self, a):
        'One preprocessed item as Core ML wants it: a PIL image for an image input, else an array.'
        if self.inp.kind != 'image': return a.astype('float32')
        from PIL import Image
        return Image.fromarray(a.astype('uint8'))

    def decode(self, outs, meta:dict, src=None, **kw) -> Pred:
        'Core ML classifiers answer with `{label: probability}`, which is already decoded.'
        d = getattr(self, '_last', None) or {}
        probs = next((v for v in d.values() if isinstance(v, dict)), None)
        if probs is not None:
            return Pred(src=src, task='classify', model=self.name,
                        preds=decode_class_dict(probs, kw.get('topk', self.topk)))
        return super().decode(outs, meta, src, **kw)

## On a Mac

Nothing above runs in CI, which is linux. On a Mac the call is the one every other runtime takes.

In [ ]:
#| eval: false
m = Model('models/MobileNetV2.mlpackage')          # or a hub repo that ships one
m('bird.jpg')

In [ ]:
#| eval: false
m.spec.kind, m.task, m.prep

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()